In [1]:
from microscope_calibration.common.model import Parameters4DSTEM, DescanError, trace, PixelYX, Model4DSTEM, Ray

from temgym_core.run import run_iter

import jax
import jax.numpy as jnp
import numpy as np
import sympy as sym
from sympy import S
from types import ModuleType

In [2]:
from sympy.abc import a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, s, t, u, v, w, x
parms= Parameters4DSTEM(
    overfocus=0.01*a,
    scan_pixel_pitch=1e-6*b,
    scan_center=PixelYX(0.1*c, 0.2*d),
    scan_rotation=0.01*e,
    camera_length=1.0*f,
    detector_pixel_pitch=50e-6*g,
    detector_center=PixelYX(0.1*h, 0.1*i),
    semiconv=1e-3*j,  # radian
    flip_factor=1.0*k,
    descan_error=DescanError(pxo_pxi=l, pxo_pyi=m, pyo_pxi=n, pyo_pyi=o, sxo_pxi=p, sxo_pyi=q, syo_pxi=r, syo_pyi=s, offpxi=t, offpyi=u, offsxi=v, offsyi=w),
)

In [3]:
def make_a_function(a: str):
    def f():
        return f'hello {a}'
    return f

In [4]:
f = make_a_function('Margarita')

In [5]:
f()

'hello Margarita'

In [6]:
def scale(factor):
    return sym.eye(2) * factor


def rotate(radians):
    return sym.rot_givens(0, 1, radians, dim=2)


# The flip_factor is introduced to make it differentiable
def flip_y(flip_factor: sym.Float = sym.S(-1.0)):
    return sym.Matrix([[flip_factor, 0], [0, 1]])


def identity():
    return sym.eye(2)

In [21]:
def scale_rotate_flip(mat: sym.Matrix):
    """
    Deconstruct a matrix generated with scale() @ rotate() @ flip_y()
    into the individual parameters
    """
    scale_y = mat[:, 0].norm()
    scale_x = mat[:, 1].norm()
    
    if not scale_x.equals(scale_y):
        raise ValueError(f"y scale {scale_y} and x scale {scale_x} are different.")

    scan_rot_flip = mat / scale_y

    # 2D cross product
    flip_factor = (
        scan_rot_flip[0, 0] * scan_rot_flip[1, 1]
        - scan_rot_flip[0, 1] * scan_rot_flip[1, 0]
    )
    # undo flip_y
    rot = scan_rot_flip.copy()
    rot[:, 0] = rot[:, 0] * flip_factor
    rot = sym.simplify(rot)

    angle1 = sym.atan2(-rot[1, 0], rot[0, 0])
    angle2 = sym.atan2(rot[0, 1], rot[1, 1])

    # So far not reached in tests since inconsistencies are caught as shear before
    if not sym.Matrix([sym.sin(angle1), sym.cos(angle1)]).equals(sym.Matrix([sym.sin(angle2), sym.cos(angle2)])):
        raise ValueError(
            f"Rotation angle 1 {angle1} and rotation angle 2 {angle2} are inconsistent."
        )

    if sym.Or(sym.And(mat[0,0].equals(mat[1,1]), mat[0,1].equals(-mat[1,0])), sym.And(mat[0,0].equals(-mat[1,1]), mat[0,1].equals(mat[1,0]))) is not sym.S.true:
        raise ValueError(f"y scale {scale_y} and x scale {scale_x} are different or rotation angle 1 {angle1} and rotation angle 2 {angle2} are inconsistent.")
        
    return (scale_y, angle1, flip_factor)

In [14]:
a, b, c, d = sym.symbols('a b c d', real=True)
expr = sym.sqrt(a**2+c**2)-sym.sqrt(b**2+d**2)
sol_c = sym.solve(expr, c)[0]
sol_d = sym.solve(expr, d)[0]
expr_subs = expr.subs(c, sol_c).subs(d, sol_d)
expr_subs
#scale_rotate_flip(sym.Matrix([[a,b],[b,a]]))

0

In [20]:
scale_rotate_flip(sym.Matrix([[0,0],[0,0]]))
sym.And(a.equals(a)) is sym.S.true

True

In [ ]:
a, b, c, d = sym.symbols('a b c d', real=True)
mat = sym.Matrix([[a,b],[c,d]])
scale_y = mat[:, 0].norm()
mat = mat/scale_y
flip = mat[0,0]*mat[1,1] - mat[0,1]*mat[1,0]
mat[:,0] = mat[:,0]*flip
sym.solve([expr, sym.sin(sym.atan2(-mat[1,0], mat[0,0]))-sym.sin(sym.atan2(mat[0,1],mat[1,1])), sym.cos(sym.atan2(-mat[1,0], mat[0,0]))-sym.cos(sym.atan2(mat[0,1],mat[1,1]))], [c, d], dict=True)

In [ ]:
def relations(a, b, eq):
    

In [ ]:
def derive(
    params: Parameters4DSTEM,
    overfocus: sym.Basic | None = None,  # m
    scan_pixel_pitch: sym.Basic | None = None,  # m
    scan_center: PixelYX | None = None,
    scan_rotation: sym.Basic | None = None,  # rad
    camera_length: sym.Basic | None = None,  # m
    detector_pixel_pitch: sym.Basic | None = None,  # m
    detector_center: PixelYX | None = None,
    detector_rotation: sym.Basic | None = None,  # rad
    semiconv: sym.Basic | None = None,  # rad
    flip_y: sym.logic.boolalg.Boolean | None = None,
    flip_factor: sym.Basic | None = None,
    descan_error: DescanError | None = None,
) -> "Parameters4DSTEM":
    if flip_factor is not None:
        assert flip_y is None
    if flip_y is not None:
        flip_factor = -1. if flip_y else 1.
            
    self = params
    
    return Parameters4DSTEM(
        overfocus=overfocus if overfocus is not None else self.overfocus,
        scan_pixel_pitch=(
            scan_pixel_pitch
            if scan_pixel_pitch is not None
            else self.scan_pixel_pitch
        ),
        scan_center=scan_center if scan_center is not None else self.scan_center,
        scan_rotation=scan_rotation
        if scan_rotation is not None
        else self.scan_rotation,
        camera_length=camera_length
        if camera_length is not None
        else self.camera_length,
        detector_pixel_pitch=(
            detector_pixel_pitch
            if detector_pixel_pitch is not None
            else self.detector_pixel_pitch
        ),
        detector_center=(
            detector_center if detector_center is not None else self.detector_center
        ),
        detector_rotation=(
            detector_rotation
            if detector_rotation is not None
            else self.detector_rotation
        ),
        semiconv=semiconv if semiconv is not None else self.semiconv,
        flip_factor=flip_factor if flip_factor is not None else self.flip_factor,
        descan_error=descan_error
        if descan_error is not None
        else self.descan_error,
    )

In [ ]:
def adjust_scan_rotation(params, scan_rotation) -> "Parameters4DSTEM":
    self = params
    de = self.descan_error
    angle = scan_rotation - self.scan_rotation

    # Rotate the input direction
    pxo_pyi, pxo_pxi = rotate(angle).multiply(sym.Matrix([de.pxo_pyi, de.pxo_pxi]))  # add simplificatoin? dotprodsimp=True
    pyo_pyi, pyo_pxi = rotate(angle).multiply(sym.Matrix([de.pyo_pyi, de.pyo_pxi]))
    sxo_pyi, sxo_pxi = rotate(angle).multiply(sym.Matrix([de.sxo_pyi, de.sxo_pxi]))
    syo_pyi, syo_pxi = rotate(angle).multiply(sym.Matrix([de.syo_pyi, de.syo_pxi]))
    new_de = DescanError(
        pxo_pyi=pxo_pyi,
        pyo_pyi=pyo_pyi,
        pxo_pxi=pxo_pxi,
        pyo_pxi=pyo_pxi,
        sxo_pyi=sxo_pyi,
        syo_pyi=syo_pyi,
        sxo_pxi=sxo_pxi,
        syo_pxi=syo_pxi,
        offpxi=de.offpxi,
        offpyi=de.offpyi,
        offsxi=de.offsxi,
        offsyi=de.offsyi,
    )
    return self.derive(
        scan_rotation=scan_rotation,
        descan_error=new_de,
    )

In [ ]:
def adjust_scan_pixel_pitch(params, scan_pixel_pitch: sym.Basic) -> "Parameters4DSTEM":
    self = params
    
    de = self.descan_error
    ratio = self.scan_pixel_pitch / scan_pixel_pitch

    new_de = DescanError(
        pxo_pyi=de.pxo_pyi * ratio,
        pyo_pyi=de.pyo_pyi * ratio,
        pxo_pxi=de.pxo_pxi * ratio,
        pyo_pxi=de.pyo_pxi * ratio,
        sxo_pyi=de.sxo_pyi * ratio,
        syo_pyi=de.syo_pyi * ratio,
        sxo_pxi=de.sxo_pxi * ratio,
        syo_pxi=de.syo_pxi * ratio,
        offpxi=de.offpxi,
        offpyi=de.offpyi,
        offsxi=de.offsxi,
        offsyi=de.offsyi,
    )
    return self.derive(
        scan_pixel_pitch=scan_pixel_pitch,
        descan_error=new_de,
    )